# nb_01_bronze_ingest — land the raw payload, exactly as received

**Bronze rules:** no business transforms; append-by-run history under
`ingest_date=…/run_id=…`; fail loudly on HTTP errors, ArcGIS error-in-200 bodies, and empty
batches (CA has a non-zero baseline — an empty batch is breakage, while empty OR/WA alone is
a data fact handled downstream).

**Exits** with a JSON payload (`run_id`, `bronze_dir`, ingest metadata) that the pipeline
passes to the Silver and Gold notebooks.

In [ ]:
storage_account = ""          # required — ADLS Gen2 account name
lake_container = "lake"
source = "live"               # "live" | "fixture" (fixture seeds both DQ paths)
run_id = ""                   # optional explicit run id (normally empty)

In [ ]:
%run nb_00_config

In [ ]:
cfg = init_config(storage_account, lake_container)


def _query_page(session, offset):
    params = {
        "where": WHERE_CLAUSE, "outFields": "*", "returnGeometry": "true",
        "f": "json", "resultOffset": offset, "resultRecordCount": PAGE_SIZE,
    }
    resp = session.get(QUERY_URL, params=params, timeout=HTTP_TIMEOUT_SECONDS)
    if resp.status_code != 200:
        raise PipelineError(f"FeatureServer returned HTTP {resp.status_code}: {resp.text[:500]}")
    payload = resp.json()
    if "error" in payload:  # ArcGIS reports many failures inside a 200 response
        raise PipelineError(f"FeatureServer returned an error payload: {payload['error']}")
    if "features" not in payload:
        raise PipelineError(f"Unexpected response shape: {list(payload.keys())}")
    return payload


def fetch_live_pages():
    pages, offset = [], 0
    with requests.Session() as session:
        while True:
            payload = _query_page(session, offset)
            pages.append(payload)
            n = len(payload["features"])
            log.info("Fetched page %d: %d features (offset=%d)", len(pages), n, offset)
            if payload.get("exceededTransferLimit") and n > 0:
                offset += n
            else:
                return pages


def load_fixture_pages():
    rows = spark.read.text(cfg.fixture_path, wholetext=True).collect()
    if not rows:
        raise PipelineError(f"Fixture not found or empty: {cfg.fixture_path}")
    return [json.loads(rows[0]["value"])]


def ingest_bronze(source, run_id=None):
    started_at = datetime.now(timezone.utc)
    run_id = run_id or new_run_id(started_at)
    ingest_date = started_at.strftime("%Y-%m-%d")

    if source == "live":
        pages, source_detail = fetch_live_pages(), QUERY_URL
    elif source == "fixture":
        pages, source_detail = load_fixture_pages(), cfg.fixture_path
    else:
        raise PipelineError(f"Unknown source '{source}' (expected 'live' or 'fixture')")

    rows_in = sum(len(p["features"]) for p in pages)
    if rows_in == 0:
        raise PipelineError(
            "FeatureServer returned 0 features for CA/OR/WA — refusing to land an empty "
            "bronze batch (empty OR/WA is a data fact, an empty batch is not)."
        )

    bronze_dir = f"{cfg.bronze_root}/ingest_date={ingest_date}/run_id={run_id}"
    for i, payload in enumerate(pages, start=1):
        mssparkutils.fs.put(f"{bronze_dir}/page_{i:04d}.json", json.dumps(payload), True)
    metadata = {
        "run_id": run_id, "ingest_date": ingest_date,
        "ingested_at_utc": started_at.isoformat(), "source": source,
        "source_detail": source_detail, "where_clause": WHERE_CLAUSE,
        "pages": len(pages), "rows_in": rows_in,
    }
    mssparkutils.fs.put(f"{bronze_dir}/_ingest_metadata.json", json.dumps(metadata, indent=2), True)
    log.info("Bronze landed: %s (%d rows, %d page(s))", bronze_dir, rows_in, len(pages))
    return run_id, bronze_dir, metadata


effective_run_id, bronze_dir, metadata = ingest_bronze(source, run_id or None)
exit_value = json.dumps({"run_id": effective_run_id, "bronze_dir": bronze_dir, **metadata})
print(exit_value)
mssparkutils.notebook.exit(exit_value)